# Start of the Sensitivity Analysis of the RCMs 

Program to read in the zarr collection of an RCM and its corresponding FFDI files and explore FFDI drivers

This code reads in the atmospheric data, computes the DI index and then does the correlations between atm
and FFDI for the 95% of FFDI


FFDI code needs the following kernel!
3-23-.10




#### required packages

In [ ]:
import intake
import xarray as xr
from matplotlib import pyplot as plt
import numpy as np

from glob import glob
import pathlib
import traceback
from datetime import datetime

from xclim.indices import (
    keetch_byram_drought_index,
    griffiths_drought_factor,
    mcarthur_forest_fire_danger_index
)

%load_ext autoreload
%autoreload 2


# importing sys
import sys
# adding plotting module to the system path
sys.path.insert(0, '/g/data/xv83/rxm599/acs/plotting_maps')
# import ACS plotting maps and Xarray.
from acs_plotting_maps import *
from acs_area_statistics import acs_regional_stats, get_regions

import geopandas as gpd
import pandas as pd
import regionmask

sys.path.insert(0, '/g/data/xv83/rxm599/acs/hazard_fire/paper2')
import sa as sa
import df as df


#### start a local Dask client

In [ ]:
from dask.distributed import Client, LocalCluster
import dask
import os

# --- Dask optimiser-style settings ---
dask.config.set({
    'distributed.comm.timeouts.connect': '90s',  # Timeout for connecting to a worker
    'distributed.comm.timeouts.tcp': '90s',  # Timeout for TCP communications
#    "distributed.worker.memory.target": False,
#    "distributed.worker.memory.spill": False,
#    "distributed.worker.memory.pause": False,
#    "distributed.worker.memory.terminate": False,
})

# --- Match ARE allocation ---
ncpus = int(os.environ.get("PBS_NCPUS", 1))

cluster = LocalCluster(
    n_workers=ncpus,      # one worker per CPU
    threads_per_worker=1, # critical (matches optimiser)
    processes=True,
    memory_limit=0        # removes worker memory limit
)

client = Client(cluster)
client

In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Set parameters

In [ ]:
# parameters
# 0,2,3 failed
mindex=2
#lon1=140; lon2=150 
#lat1=-45; lat2= -32
#lat1=-45 lat2= -40 lon1=144 lon2=149
lat1=-45
lat2= -10
lon1=110
lon2=155
p_ext=0.95
p_ext=0.99

#lon1=145 ; lon2=146
#lat1=-37 ; lat2=-36
t1='2015-01-01'
t2='2035-01-01'
dirstore='sa_2020'
#dirstore='sa_obs'

In [ ]:
# extra info
mindexp100=mindex+100
mobs=39

## Obtain the desired catalogues of the simulations to processe

In [ ]:
dtype='/g/data/ia39/ncra/fire/bias-output/'
zarr_path=dtype+ 'zarr/*ssp370*'
ffdi_path=dtype+'ffdi/*ssp370*FFDI*'
kbdi_path=dtype+'ffdi/*ssp370*KBDI*'
# observations 
zobs='/g/data/ia39/ncra/fire/bias-input/zarr/*ERA*' 
fobs='/g/data/ia39/ncra/fire/bias-input/ffdi/*ERA*FFDI*'
kobs='/g/data/ia39/ncra/fire/bias-input/ffdi/*ERA*KBDI*'

mRuns = sorted(glob(zarr_path)) + sorted(glob(zobs))
mFFDI = sorted(glob(ffdi_path))+ sorted(glob(fobs))
mKBDI = sorted(glob(kbdi_path)) + sorted(glob(kobs))
print(len(mRuns))
print(len(mFFDI))

# print code to ensure the files match
ifile=-1
for file in mRuns: 
    ifile=ifile+1
    print(ifile, file)
ifile=-1
for file in mFFDI: 
    ifile=ifile+1
#    print(ifile, file)
   

# Process one ensemble 

In [ ]:
# From one catalogue list save variables
ds0=xr.open_zarr(mRuns[mindex])
df0=xr.open_zarr(mFFDI[mindex])
dk0=xr.open_zarr(mKBDI[mindex])

print(mRuns[mindex])
print(mFFDI[mindex])
print(mKBDI[mindex])


# compute DF 
# kbdi has a 20 extra points at the start for DF calculation
if mindex != mobs: 
# for projections    
    pra = ds0.prAdjust.sel(time=slice(ds0.prAdjust.time[0],'2099-12-31'))
    pra = pra.transpose("lat", "lon","time")
#    pra = ds0.prAdjust.sel(time=slice(ds0.prAdjust.time[0],'2100-01-01')) # kbdi does not have data beyond end of december
    kbdi=dk0.KBDI.sel(time=slice(pra.time[0],pra.time[-1] ))
# xclim expects time to be the core dimension for drought indices
# Reorder KBDI to match (time, lat, lon)
#    kbdi = KBDI.transpose("time", "lat", "lon")
    DF = df.griffiths_drought_factor_dask_exact(pra, kbdi)
else:
    print("obs")
    pra = ds0.pr.sel(time=slice(ds0.pr.time[0],'2100-01-01'))
#    pra = pra1.transpose("lat", "lon","time")
    kbdi=dk0.KBDI.sel(time=slice(pra.time[0],pra.time[-1] ))
# Reorder KBDI to match (time, lat, lon)
#    kbdi = KBDI.transpose("time", "lat", "lon")
    DF = df.griffiths_drought_factor_dask_exact(pra, kbdi)
    
#print(pra)
#print(kbdi)
dk0

In [ ]:
# compute the 95% value from first 20 years
if mindex != mobs:
    tr1='2015-01-01'; tr2='2035-01-01'
# should be changed to     tr1='2015-01-01'; tr2='2035-01-01'
else:
    tr1='2000-01-01'; tr2='2020-01-01'
    t1=tr1; t2=tr2
# should be changed to tr1='2004-01-01'; tr2='2024-01-01' 
#    tr1='2003-01-01'; tr2='2023-12-31'

df1=df0.sel(time=slice(tr1,tr2))
d95a=df1.FFDI.quantile(p_ext,dim='time')


### test output   marked up for the moment.

dd2=DF.sel(time=slice(t1,t2))
dd2.to_zarr('/scratch/xv83/rxm599/crap.zarr',mode='w',zarr_format=2)



pra = pra.where(pra >= 0)
kk_mm = kk_mm.where(kk_mm >= 0)

pra = pra.fillna(0)
kk_mm = kk_mm.fillna(0)

pra0 = pra.isel(lat=0, lon=0).compute()
kk0  = kk_mm.isel(lat=0, lon=0).compute()

print(np.isnan(pra0).sum(), np.isnan(kk0).sum())
print((pra0 < 0).sum(), (kk0<0).sum())

# original

## mask info

In [ ]:
%%time
# NCRA regions from acs_area_statistics code
# these are the names of your regions
regions = get_regions([
                           "australia"
                      ])
regions
# nrm_regions ncra_regions",


In [ ]:
mask_frac = regions.mask_3D(d95a)
#mask_frac = regions.mask_3D_frac_approx(d95) # not defined in 3-23.10

In [ ]:
mask_ = mask_frac.isel(region=0)
mask_
d95=d95a.where(mask_).load()

In [ ]:
d95.plot(cmap='plasma',levels=10)

In [ ]:
df2=df0.sel(time=slice(t1,t2))
dfe95=df0.FFDI.where(df2.FFDI > d95)


In [ ]:
df2=df0.sel(time=slice(t1,t2))
ds2=ds0.sel(time=slice(t1, t2))
dk2=dk0.sel(time=slice(t1,t2))
dd2=DF.sel(time=slice(t1,t2))
# extract only the values greater than FFDI > 95%
dfe95=df0.FFDI.where(df2.FFDI > d95)
dse95=ds0.where(df2.FFDI > d95)
dke95=dk0.where(df2.FFDI > d95)
dde95=DF.where(df2.FFDI > d95)
print(dfe95)
print(dse95)
print(dke95)
print(dde95)

In [ ]:
dd2

In [ ]:
# fixed naming in UQ-DEC files
print(list(dse95.data_vars))
var=list(dse95.data_vars)
for jj in list(dse95.data_vars):
    if (jj == 'sfcWindAdjust'):
        qt=True
        ddnew=dse95.rename({'sfcWindAdjust': 'sfcWindmaxAdjust'})
        dse95=ddnew

print(list(dse95.data_vars))

In [ ]:
%%time
FT=dfe95
# compute seasonal diagnostics for subsequent use
mask=FT.notnull()
days = xr.where(mask, 1, np.nan)
tot_days=(days.groupby('time.season').sum('time')+d95*0).load()
seasonal_clim = (FT.groupby('time.season').mean('time')).load()


In [ ]:
%%time
outf='/scratch/xv83/rxm599/'+f'{dirstore}/'+f'model_{mindexp100}_ffdi_ex.nc'
outf1='/scratch/xv83/rxm599/'+f'{dirstore}/'+f'model_{mindexp100}_ffdi_t.zarr'

d95.name='FFDI_mean'
dfe95.name='FFDI_t'
tot_days.name='tot_days'
seasonal_clim.name='seasonal_clim'

# compute the p-ext% value for the 20 years of the analysis
ffdi0=df0.sel(time=slice(t1,t2))
ffdi1=ffdi0.FFDI.quantile(p_ext,dim='time')
ffdi2=ffdi1.where(mask_).load()
ffdi2.name='FFDI_mean'

# compare reference date (tr1) with start date of analysis (t1)
if t1 == tr1 :
    print("Times are equal")
    dfout=xr.merge([d95,tot_days,seasonal_clim])
else:
    print("Times are NOT equal")
    dfout=xr.merge([ffdi2,tot_days,seasonal_clim])

# Only create file if it does not exist
print(outf)
if os.path.exists(outf):
    print("File exists!",outf)
    ddtmp1=xr.open_dataset(outf)
else:
    print("File does not exist. ",outf)
    dfout.to_netcdf(outf, mode='w')
    ddtmp1=xr.open_dataset(outf)
    
if os.path.exists(outf1):
    print("File exists!",outf1)
    ddtmp2=xr.open_zarr(outf1)
else:
    print("File does not exist. ",outf1)
    dfe95.to_zarr(outf1, mode='w',zarr_format=2)
    ddtmp2=xr.open_zarr(outf1)




In [ ]:
%%time
# Reduce workers for IO phase

# Only create file if it does not exist
outf='/scratch/xv83/rxm599/'+f'{dirstore}/'+f'model_{mindexp100}_dde95.zarr'
outft='/scratch/xv83/rxm599/'+f'{dirstore}/'+f'model_{mindexp100}_dde95.nc'
print(outf)
#dde95.load()
if os.path.exists(outf):
    print("File exists!")
    ddtmp=xr.open_zarr(outf)
else:
    print("File does not exist.")
#    dde95.to_zarr(outf, mode='w',encoding=encoding)
    dde95.to_zarr(outf, mode='w',zarr_format=2)
#    dfe95.to_netcdf(outft, mode='w')
    ddtmp=xr.open_zarr(outf)


In [ ]:
#print(dde95)
var_name = list(ddtmp.data_vars)[0]
dde95 = ddtmp[var_name]
#dde95.mean('time').plot()
dde95

# Correlation analysis with whole dataset

In [ ]:
%%time 
# compute the average of the environmnental variables for the days when FFDI > 95% (or p_ext%)
outf='/scratch/xv83/rxm599/'+f'{dirstore}/'+f'model_{mindexp100}_ffdi_atm.nc'
print(outf)

dse95

dde95

In [ ]:
%%time 
# compute the average of the environmnental variables for the days when FFDI > 95% (or p_ext%)
outf='/scratch/xv83/rxm599/'+f'{dirstore}/'+f'model_{mindexp100}_ffdi_atm.nc'
print(outf)

if mindex != mobs:
    t_ave=dse95.tasmaxAdjust.mean(dim='time')
    h_ave=dse95.hursminAdjust.mean(dim='time')
    w_ave=dse95.sfcWindmaxAdjust.mean(dim='time')
    DF_ave=dde95.mean(dim='time')
else:
    t_ave=dse95.tasmax.mean(dim='time')
    h_ave=dse95.hursmin.mean(dim='time')
    w_ave=dse95.sfcWindmax.mean(dim='time')
    DF_ave=dde95.mean(dim='time')
#t_ave.name='tasmax'
#h_ave.name='hursmin'
#w_ave.name='sfcWindmax'
#DF_ave.name='DF'
dout=xr.merge([t_ave,h_ave,w_ave,DF_ave])

if os.path.exists(outf):
    print("File exists!")
else:    
    dout.to_netcdf(outf, mode='w')



In [ ]:
%%time
# two versions 
if mindex != mobs: 
# for projections    
    r1=xr.corr(dse95.tasmaxAdjust,dfe95,dim='time').load()
    r2=xr.corr(dse95.hursminAdjust,dfe95,dim='time').load()
    r3=xr.corr(dde95,dfe95,dim='time').load()
    r4=xr.corr(dse95.sfcWindmaxAdjust,dfe95,dim='time').load()
else:
    r1=xr.corr(dse95.tasmax,dfe95,dim='time').load()
    r2=xr.corr(dse95.hursmin,dfe95,dim='time').load()
    r3=xr.corr(dde95,dfe95,dim='time').load()
    r4=xr.corr(dse95.sfcWindmax,dfe95,dim='time').load()
 

In [ ]:
%%time
#Correlation with FFDI
outf1='/scratch/xv83/rxm599/'+f'{dirstore}/'+f'model_{mindexp100}_corr.nc'
print(mindex)
print(outf1)
if os.path.exists(outf1):
    print("File exists!")
else:
    print("File does not exist.")
    r1.name='r1'
    r2.name='r2'
    r3.name='r3'
    r4.name='r4'
    ds = xr.merge([r1,r2,r3,r4])
    ds.to_netcdf(outf1)



In [ ]:
%%time
lev1=np.arange(-.80, 1, .20)
plt.figure(figsize=(12, 12))
plt.subplot(2,2,1); r1.plot(levels=lev1,extend='both')
plt.title("Tmax - FFDI")
plt.subplot(2,2,2); r2.plot(levels=lev1,extend='both')
plt.title("Hmin - FFDI")
plt.subplot(2,2,3); r3.plot(levels=lev1,extend='both')
plt.title("DF - FFDI")
plt.subplot(2,2,4); r4.plot(levels=lev1,extend='both')
plt.title("Wmax - FFDI")


In [ ]:
%%time
# correlations between FFDI variables
if mindex != mobs: 
# for projections    
    r1=xr.corr(dde95,dse95.hursminAdjust,dim='time').load()
    r2=xr.corr(dde95,dse95.tasmaxAdjust,dim='time').load()
    r3=xr.corr(dde95,dse95.sfcWindmaxAdjust,dim='time').load()
    r4=xr.corr(dse95.tasmaxAdjust,dse95.hursminAdjust,dim='time').load()
    r5=xr.corr(dse95.tasmaxAdjust,dse95.sfcWindmaxAdjust,dim='time').load()
    r6=xr.corr(dse95.sfcWindmaxAdjust,dse95.hursminAdjust,dim='time').load()
else:
    r1=xr.corr(dde95,dse95.hursmin,dim='time').load()
    r2=xr.corr(dde95,dse95.tasmax,dim='time').load()
    r3=xr.corr(dde95,dse95.sfcWindmax,dim='time').load()
    r4=xr.corr(dse95.tasmax,dse95.hursmin,dim='time').load()
    r5=xr.corr(dse95.tasmax,dse95.sfcWindmax,dim='time').load()
    r6=xr.corr(dse95.sfcWindmax,dse95.hursmin,dim='time').load()
 

In [ ]:
%%time
# check correlation between input variables to FFDI
plt.figure(figsize=(8,14))
lev1=np.arange(-.80, 1, .20)
#lev1=np.arange(-.90, 1, .10)
plt.subplot(3,2,1); r1.plot(levels=lev1,extend='both')
plt.title("DI - Tmax")
plt.subplot(3,2,2); r2.plot(levels=lev1,extend='both')
plt.title("DI - Hmin")
plt.subplot(3,2,3); r3.plot(levels=lev1,extend='both')
plt.title("DI - Wmax")
plt.subplot(3,2,4); r4.plot(levels=lev1,extend='both')
plt.title("Tmax - Hmin")
plt.subplot(3,2,5); r5.plot(levels=lev1,extend='both')
plt.title("Tmax - Wmax")
plt.subplot(3,2,6); r6.plot(levels=lev1,extend='both')
plt.title("Hmin - Wmax`")

In [ ]:
%%time
outf1='/scratch/xv83/rxm599/'+f'{dirstore}/'+f'model_{mindexp100}_xcorr.nc'
print(mindex)
print(outf1)

if os.path.exists(outf1):
    print("File exists!")
else:
    print("File does not exist.")
    r1.name='r1'
    r2.name='r2'
    r3.name='r3'
    r4.name='r4'
    r5.name='r5'
    r6.name='r6'
    ds = xr.merge([r1,r2,r3,r4,r5,r6])
    ds.to_netcdf(outf1)



In [ ]:
#client.shutdown()